下面是一份可直接集成到训练循环中的监控脚本框架，支持每 N 步自动记录指标、计算分位数、判断健康状态并输出告警。

# 核心监控类 TrainMonitor

In [ ]:
import numpy as np
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Dict, Optional

@dataclass
class ThresholdConfig:
    """各指标的健康/警告阈值配置"""
    # 奖励分布
    reward_std_min: float = 0.1
    reward_range_ratio_min: float = 0.1      # p5~p95 区间占全量程的最小比例
    reward_tail_ratio_max: float = 5.0        # p99/p50 最大比值
    zero_var_group_ratio_max: float = 0.3     # 零方差组占比上限

    # 概率比 ratio
    ratio_p50_low: float = 0.9
    ratio_p50_high: float = 1.1
    ratio_p99_max: float = 10.0
    ratio_p1_min: float = 0.1

    # 优势值
    advantage_std_min: float = 0.05
    advantage_p99_max: float = 10.0

    # KL 散度
    kl_mean_max: float = 0.05
    kl_p95_max: float = 0.05

    # 梯度范数
    grad_norm_p50_low: float = 0.01
    grad_norm_p50_high: float = 5.0
    grad_norm_p99_max: float = 10.0

    # 策略熵
    entropy_ratio_min: float = 0.2            # 熵不低于初始值的 20%

    # 基础设施
    max_consecutive_nan_steps: int = 0


class TrainMonitor:
    """
    训练监控器：每 N 步记录指标、计算分位数、判断健康状态并输出告警。

    用法：
        monitor = TrainMonitor(log_every=50, warmup_steps=200)

        for step in range(total_steps):
            # ... 训练一步 ...
            monitor.update(step=step, metrics={
                "rewards": batch_rewards,           # np.array
                "ratios": batch_ratios,             # np.array
                "advantages": batch_advantages,     # np.array
                "kl": batch_kl,                     # np.array 或 float
                "grad_norm": grad_norm,             # float
                "policy_entropy": entropy,          # float
                "zero_var_group_ratio": zv_ratio,   # float
                "trainable_params": num_params,     # int
                "has_nan": has_nan_in_weights,      # bool
            })
    """

    def __init__(
        self,
        log_every: int = 50,
        warmup_steps: int = 200,
        thresholds: Optional[ThresholdConfig] = None,
        logger=None,
    ):
        self.log_every = log_every
        self.warmup_steps = warmup_steps
        self.thresholds = thresholds or ThresholdConfig()
        self.logger = logger  # 可传入 wandb / tensorboard writer

        # 内部状态
        self._history = defaultdict(list)       # 存储各指标的历史记录
        self._baseline = {}                     # warmup 阶段建立的健康基线
        self._initial_entropy = None            # 初始策略熵
        self._consecutive_nan_steps = 0         # 连续 NaN 步数
        self._step_count = 0

    def update(self, step: int, metrics: Dict):
        """每步调用，传入当前步的各项指标原始数据"""
        self._step_count = step

        # 1. 计算分位数统计量
        stats = self._compute_stats(metrics)

        # 2. 记录历史
        for k, v in stats.items():
            self._history[k].append(v)

        # 3. warmup 阶段：建立基线
        if step <= self.warmup_steps:
            if step == self.warmup_steps:
                self._build_baseline()
                print(f"[Monitor] 基线建立完成 (warmup={self.warmup_steps} 步)")
            return

        # 4. 记录初始熵
        if self._initial_entropy is None and "policy_entropy" in metrics:
            self._initial_entropy = float(np.mean(metrics["policy_entropy"]))

        # 5. 每 N 步执行健康检查
        if step % self.log_every == 0:
            alerts = self._check_health(stats)
            self._log(step, stats, alerts)

    def _compute_stats(self, metrics: Dict) -> Dict:
        """从原始指标计算分位数等统计量"""
        stats = {}

        # --- 奖励分布 ---
        if "rewards" in metrics:
            r = np.asarray(metrics["rewards"], dtype=np.float64)
            stats["reward_mean"] = float(np.mean(r))
            stats["reward_std"] = float(np.std(r))
            stats["reward_p5"] = float(np.percentile(r, 5))
            stats["reward_p50"] = float(np.percentile(r, 50))
            stats["reward_p95"] = float(np.percentile(r, 95))
            stats["reward_p99"] = float(np.percentile(r, 99))
            r_range = stats["reward_p95"] - stats["reward_p5"]
            r_full = (r.max() - r.min()) if r.max() != r.min() else 1.0
            stats["reward_range_ratio"] = r_range / r_full

        # --- 概率比 ratio ---
        if "ratios" in metrics:
            ratio = np.asarray(metrics["ratios"], dtype=np.float64)
            stats["ratio_p1"] = float(np.percentile(ratio, 1))
            stats["ratio_p5"] = float(np.percentile(ratio, 5))
            stats["ratio_p50"] = float(np.percentile(ratio, 50))
            stats["ratio_p95"] = float(np.percentile(ratio, 95))
            stats["ratio_p99"] = float(np.percentile(ratio, 99))

        # --- 优势值 ---
        if "advantages" in metrics:
            adv = np.asarray(metrics["advantages"], dtype=np.float64)
            stats["advantage_mean"] = float(np.mean(adv))
            stats["advantage_std"] = float(np.std(adv))
            stats["advantage_p10"] = float(np.percentile(adv, 10))
            stats["advantage_p90"] = float(np.percentile(adv, 90))
            stats["advantage_p99"] = float(np.percentile(adv, 99))

        # --- KL 散度 ---
        if "kl" in metrics:
            kl = np.asarray(metrics["kl"], dtype=np.float64)
            stats["kl_mean"] = float(np.mean(kl))
            stats["kl_p50"] = float(np.percentile(kl, 50))
            stats["kl_p95"] = float(np.percentile(kl, 95))

        # --- 梯度范数 ---
        if "grad_norm" in metrics:
            gn = metrics["grad_norm"]
            if isinstance(gn, (int, float)):
                stats["grad_norm"] = float(gn)
            else:
                gn = np.asarray(gn, dtype=np.float64)
                stats["grad_norm"] = float(np.mean(gn))
                stats["grad_norm_p99"] = float(np.percentile(gn, 99))

        # --- 策略熵 ---
        if "policy_entropy" in metrics:
            stats["policy_entropy"] = float(
                np.mean(metrics["policy_entropy"])
                if hasattr(metrics["policy_entropy"], '__len__')
                else metrics["policy_entropy"]
            )

        # --- 零方差组占比 ---
        if "zero_var_group_ratio" in metrics:
            stats["zero_var_group_ratio"] = float(metrics["zero_var_group_ratio"])

        # --- 基础设施 ---
        if "trainable_params" in metrics:
            stats["trainable_params"] = int(metrics["trainable_params"])
        if "has_nan" in metrics:
            stats["has_nan"] = bool(metrics["has_nan"])

        return stats

    def _build_baseline(self):
        """用 warmup 阶段的均值作为健康基线"""
        for k, v_list in self._history.items():
            self._baseline[k] = float(np.mean(v_list))

    def _check_health(self, stats: Dict) -> list:
        """检查各项指标是否异常，返回告警列表"""
        alerts = []
        t = self.thresholds

        # --- NaN 检测 ---
        if stats.get("has_nan", False):
            self._consecutive_nan_steps += 1
            alerts.append(
                f"🔴 [NaN] 检测到权重 NaN！连续 {self._consecutive_nan_steps} 步"
            )
        else:
            self._consecutive_nan_steps = 0

        # --- 奖励分布 ---
        if "reward_std" in stats:
            if stats["reward_std"] < t.reward_std_min:
                alerts.append(
                    f"🟡 [奖励] std={stats['reward_std']:.4f} < {t.reward_std_min}，"
                    f"模型输出可能同质化"
                )
            if stats.get("reward_range_ratio", 1.0) < t.reward_range_ratio_min:
                alerts.append(
                    f"🟡 [奖励] p5~p95 区间占比={stats['reward_range_ratio']:.2%}，"
                    f"零方差组占比可能过高"
                )
            if stats["reward_p50"] != 0:
                tail_ratio = abs(stats["reward_p99"] / stats["reward_p50"])
                if tail_ratio > t.reward_tail_ratio_max:
                    alerts.append(
                        f"🟡 [奖励] 长尾比值 p99/p50={tail_ratio:.1f}，"
                        f"存在异常值驱动风险"
                    )
        if stats.get("zero_var_group_ratio", 0) > t.zero_var_group_ratio_max:
            alerts.append(
                f"🟡 [零方差组] 占比={stats['zero_var_group_ratio']:.1%}，"
                f"大量样本无法提供学习信号"
            )

        # --- 概率比 ratio ---
        if "ratio_p50" in stats:
            if not (t.ratio_p50_low <= stats["ratio_p50"] <= t.ratio_p50_high):
                alerts.append(
                    f"🟡 [Ratio] p50={stats['ratio_p50']:.3f} 偏离 1.0，"
                    f"策略更新幅度异常"
                )
            if stats["ratio_p99"] > t.ratio_p99_max:
                alerts.append(
                    f"🔴 [Ratio] p99={stats['ratio_p99']:.1f}，"
                    f"数值溢出风险！"
                )
            if stats.get("ratio_p1", 1.0) < t.ratio_p1_min:
                alerts.append(
                    f"🟡 [Ratio] p1={stats['ratio_p1']:.3f}，"
                    f"新策略抛弃了旧策略的部分行为"
                )

        # --- 优势值 ---
        if "advantage_std" in stats:
            if stats["advantage_std"] < t.advantage_std_min:
                alerts.append(
                    f"🟡 [Advantage] std={stats['advantage_std']:.4f}，"
                    f"组内差异不足，梯度可能消失"
                )
            if stats["advantage_p99"] > t.advantage_p99_max:
                alerts.append(
                    f"🟡 [Advantage] p99={stats['advantage_p99']:.2f}，"
                    f"极端样本可能绑架梯度"
                )

        # --- KL 散度 ---
        if "kl_mean" in stats:
            if stats["kl_mean"] > t.kl_mean_max:
                alerts.append(
                    f"🔴 [KL] mean={stats['kl_mean']:.4f}，"
                    f"策略偏离参考模型过远！"
                )
            elif stats.get("kl_p95", 0) > t.kl_p95_max:
                alerts.append(
                    f"🟡 [KL] p95={stats['kl_p95']:.4f}，过优化风险"
                )

        # --- 梯度范数 ---
        if "grad_norm" in stats:
            gn = stats["grad_norm"]
            if gn < t.grad_norm_p50_low:
                alerts.append(
                    f"🟡 [梯度] norm={gn:.4f}，梯度消失风险"
                )
            elif gn > t.grad_norm_p50_high:
                alerts.append(
                    f"🟡 [梯度] norm={gn:.2f}，梯度偏大"
                )
        if "grad_norm_p99" in stats:
            if stats["grad_norm_p99"] > t.grad_norm_p99_max:
                alerts.append(
                    f"🔴 [梯度] p99={stats['grad_norm_p99']:.1f}，梯度爆炸风险！"
                )

        # --- 策略熵 ---
        if "policy_entropy" in stats and self._initial_entropy is not None:
            entropy_ratio = stats["policy_entropy"] / self._initial_entropy
            if entropy_ratio < t.entropy_ratio_min:
                alerts.append(
                    f"🔴 [熵] 当前熵={stats['policy_entropy']:.3f}，"
                    f"仅为初始值的 {entropy_ratio:.0%}，熵坍塌！"
                )

        return alerts

    def _log(self, step: int, stats: Dict, alerts: list):
        """输出日志并写入外部 logger"""
        # 控制台输出
        status = "🟢 健康" if not alerts else f"⚠️ {len(alerts)} 项告警"
        print(f"\n{'='*60}")
        print(f"[Step {step}] 监控报告 — {status}")
        print(f"{'='*60}")

        # 打印关键指标摘要
        summary_keys = [
            "reward_mean", "reward_std",
            "ratio_p50", "ratio_p99",
            "advantage_std",
            "kl_mean", "kl_p95",
            "grad_norm",
            "policy_entropy",
        ]
        for k in summary_keys:
            if k in stats:
                print(f"  {k}: {stats[k]:.4f}")

        # 打印告警
        if alerts:
            print(f"\n{'─'*40}")
            print("告警详情：")
            for a in alerts:
                print(f"  {a}")
        print(f"{'='*60}\n")

        # 写入外部 logger（wandb / tensorboard）
        if self.logger is not None:
            self.logger.log(stats, step=step)
            if alerts:
                # wandb 示例：self.logger.alert(title="训练告警", text="\n".join(alerts))
                pass

## 在训练循环中集成

In [ ]:
# ===== 初始化 =====
monitor = TrainMonitor(
    log_every=50,          # 每 50 步输出一次报告
    warmup_steps=200,      # 前 200 步用于建立基线
    # logger=wandb.run,    # 可选：传入 wandb 或 tensorboard writer
)

# ===== 训练循环 =====
for step in range(total_steps):

    # --- 执行一步训练 ---
    batch_rewards, batch_ratios, batch_advantages, batch_kl = train_step(...)
    grad_norm = compute_grad_norm(model)
    entropy = compute_policy_entropy(model)

    # --- 上报指标 ---
    monitor.update(step=step, metrics={
        "rewards": batch_rewards,
        "ratios": batch_ratios,
        "advantages": batch_advantages,
        "kl": batch_kl,
        "grad_norm": grad_norm,
        "policy_entropy": entropy,
        "zero_var_group_ratio": compute_zero_var_ratio(batch_rewards),
        "trainable_params": count_trainable_params(model),
        "has_nan": check_nan_in_weights(model),
    })

## 辅助函数示例

In [ ]:
def compute_zero_var_ratio(rewards_grouped):
    """计算零方差组占比（按 Prompt 分组后的奖励）"""
    total_groups = len(rewards_grouped)
    if total_groups == 0:
        return 0.0
    zero_var_count = sum(
        1 for group in rewards_grouped if np.std(group) < 1e-8
    )
    return zero_var_count / total_groups

def check_nan_in_weights(model):
    """检查模型权重中是否存在 NaN"""
    for param in model.parameters():
        if param.grad is not None and torch.isnan(param.grad).any():
            return True
    return False

def compute_grad_norm(model):
    """计算全局梯度范数"""
    total_norm = 0.0
    for param in model.parameters():
        if param.grad is not None:
            total_norm += param.grad.data.norm(2).item() ** 2
    return total_norm ** 0.5

## 告警等级说明

| 等级 | 含义 | 建议动作 |
|------|------|----------|
| 🟢 健康 | 所有指标正常 | 继续训练 |
| 🟡 警告 | 1~2 个指标进入警告区间 | 持续观察，若连续 50 步未恢复则考虑干预 |
| 🔴 危险 | 关键指标异常（NaN/梯度爆炸/熵坍塌） | 立即暂停训练，排查工程层问题 |


## 扩展建议
 - 自定义阈值：
   - 通过修改 ThresholdConfig 中的参数适配不同模型规模。
 - 接入 WandB：
   - 将 logger 设为 wandb.run，指标会自动同步到 WandB 面板。
 - 自动早停
   - ：在 _check_health 中增加返回信号，连续 N 次红灯时自动保存 checkpoint 并终止训练。
 - 滑动窗口：
   - 将 _history 改为固定长度的 deque，避免长时间训练时内存膨胀。